# Vilier Colab Runner

Notebook nay mount Google Drive, clone/pull repo, cai dependencies, chay `bash run.sh` voi input lay tu Drive va output ghi lai Drive.

In [25]:
# Sua cac gia tri nay truoc khi chay neu can.
REPO_URL = "https://github.com/ngocbao220/vilier.git"
BRANCH = "main"
PROJECT_DIR = "/content/vilier"

# Dat file audio trong Google Drive, vi du: MyDrive/vilier/input/real.wav
DRIVE_AUDIO_PATH = "/content/drive/MyDrive/VDT-TurnTaking/inputs/real.wav"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/VDT-TurnTaking/outputs"

# Bat ASR neu muon chay PhoWhisper tren GPU Colab. Mac dinh giu false de tach speaker truoc.
ENABLE_ASR = False
ENABLE_STATE_LABELING = False
DRY_RUN = False
DIARIZATION_BACKEND = "sortformer"  # Hỗ trợ: "sortformer" hoặc "pixit"


In [26]:
from google.colab import drive
drive.mount('/content/drive')


In [27]:
import os
import subprocess
from pathlib import Path

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print("Repo:", project_dir)


In [28]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [29]:
import json
import os
import sys
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def set_secret_from_colab(secret_name, env_name):
    if os.environ.get(env_name) or userdata is None:
        return
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[env_name] = value

set_secret_from_colab("HUGGINGFACE_TOKEN", "HUGGINGFACE_TOKEN")
set_secret_from_colab("HF_TOKEN", "HF_TOKEN")
set_secret_from_colab("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY")

audio_path = Path(DRIVE_AUDIO_PATH)
output_dir = Path(DRIVE_OUTPUT_DIR)
if not audio_path.exists():
    raise FileNotFoundError(f"Drive audio not found: {audio_path}")
output_dir.mkdir(parents=True, exist_ok=True)

with open("config.json", "r", encoding="utf-8") as handle:
    config = json.load(handle)

config.setdefault("entrypoint", {})["input_path"] = str(audio_path)
config.setdefault("entrypoint", {})["output_path"] = str(output_dir)
config.setdefault("runtime", {})["dry_run"] = bool(DRY_RUN)
config.setdefault("diarization", {})["device"] = "cuda"
    config["diarization"]["backend"] = DIARIZATION_BACKEND
    if DIARIZATION_BACKEND == "sortformer":
        config["diarization"]["model"] = "nvidia/diar_sortformer_4spk-v1"
    elif DIARIZATION_BACKEND == "pixit":
        config["diarization"]["model"] = "pyannote/speech-separation-ami-1.0"
config.setdefault("asr", {})["enabled"] = bool(ENABLE_ASR)
config.setdefault("asr", {})["device"] = 0 if ENABLE_ASR else "cpu"
config.setdefault("state_labeling", {})["enabled"] = bool(ENABLE_STATE_LABELING)

config_path = Path("config.colab.json")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["CONFIG_PATH"] = str(config_path)
os.environ["INPUT_PATH"] = str(audio_path)
os.environ["OUTPUT_PATH"] = str(output_dir)
os.environ["PYTHON_BIN"] = sys.executable
if DRY_RUN:
    os.environ["DRY_RUN"] = "1"

print("CONFIG_PATH=", os.environ["CONFIG_PATH"])
print("INPUT_PATH=", os.environ["INPUT_PATH"])
print("OUTPUT_PATH=", os.environ["OUTPUT_PATH"])
print("ENABLE_ASR=", ENABLE_ASR)
print("ENABLE_STATE_LABELING=", ENABLE_STATE_LABELING)
DIARIZATION_BACKEND = "sortformer"  # Hỗ trợ: "sortformer" hoặc "pixit"


In [30]:
!hf auth login --token YOUR_HF_TOKEN

In [31]:
!bash run.sh


In [32]:
from pathlib import Path

audio_id = Path(DRIVE_AUDIO_PATH).stem
result_dir = Path(DRIVE_OUTPUT_DIR) / audio_id
print("Result dir:", result_dir)
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(result_dir))
